In [1]:
import os
import google.generativeai as genai
import requests
import re
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
import faiss
import tqdm as notebook_tqdm


C:\Users\trist\PycharmProjects\RAG-Studienarbeit\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
SRC_DIR = "./src/"

Create a Vector index:


In [6]:
import os
import google.generativeai as genai
import requests
import re
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
import faiss
import tqdm as notebook_tqdm

# ---------------------------
# Step 1: Load Processed Data
# ---------------------------
def load_chunks(json_path):
    """
    Load processed text chunks from a JSON file.
    Expected JSON format: a list of dictionaries with at least a "text" key.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Extract texts; adjust if your JSON structure differs
    texts = [chunk["text"] for chunk in data]
    return texts, data  # also returning the full data for metadata if needed

# ----------------------------------------
# Step 2: Generate Embeddings with Contriever
# ----------------------------------------
def compute_embeddings(texts, tokenizer, model, device="gpu", batch_size=16):
    """
    Compute embeddings for a list of texts using Contriever.
    """
    model.to(device)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            # Tokenize the batch. Adjust max_length if needed.
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            # The Contriever model returns the embeddings in last_hidden_state[:,0,:] or use a pooling strategy.
            # Here, we assume a pooling strategy is implemented (or use mean pooling).
            # For demonstration, we use mean pooling on the token embeddings (excluding padding tokens)
            token_embeddings = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)
            attention_mask = inputs["attention_mask"].unsqueeze(-1)  # shape: (batch_size, seq_len, 1)
            # Avoid division by zero; sum and average over non-padding tokens
            sum_embeddings = torch.sum(token_embeddings * attention_mask, dim=1)
            valid_tokens = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            batch_embeddings = sum_embeddings / valid_tokens
            embeddings.append(batch_embeddings.cpu().numpy())
    embeddings = np.vstack(embeddings)
    return embeddings

# ----------------------------------
# Step 3: Build a FAISS Vector Index
# ----------------------------------
def build_faiss_index(embeddings):
    """
    Build a FAISS index (using L2 distance) from embeddings.
    """
    d = embeddings.shape[1]  # dimension of embeddings
    index = faiss.IndexFlatL2(d)
    index.add(embeddings.astype('float32'))
    return index

# ---------------------------
# Main Execution
# ---------------------------
if __name__ == "__main__":
    # Path to your JSON file with processed chunks
    json_path = SRC_DIR + "processed_chunks.json"  # <-- change to your file path

    print(f"Loading processed chunks from {json_path}...")

    # Load texts and metadata
    texts, metadata = load_chunks(json_path)
    print(f"Loaded {len(texts)} chunks.")

    # Initialize Contriever model and tokenizer.
    # The model name "facebook/contriever" is used here; adjust if you have a different checkpoint.
    model_name = "facebook/contriever"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Compute embeddings for the texts
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embeddings = compute_embeddings(texts, tokenizer, model, device=device)
    print(f"Computed embeddings with shape: {embeddings.shape}")
    print("Embeddings:", embeddings)

    # Build a FAISS index
    index = build_faiss_index(embeddings)
    print("FAISS index built.")
    # Create the directory if it doesn't exist
    os.makedirs("./indices", exist_ok=True)

    # Assume 'index' is your FAISS index
    #faiss.write_index(index, "./indices/my_faiss_index.idx")

    # Example: Search the index with a query embedding (optional)
    # query = "Your sample query here"
    # query_embedding = compute_embeddings([query], tokenizer, model, device=device)
    # distances, indices = index.search(query_embedding.astype('float32'), k=5)
    # print("Top-5 results:", indices)


C:\Users\trist\PycharmProjects\RAG-Studienarbeit\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading processed chunks from ./src/processed_chunks.json...
Loaded 44 chunks.
Computed embeddings with shape: (44, 768)
Embeddings: [[-0.03383938 -0.02329308 -0.00130304 ... -0.02377633  0.02676156
   0.06329703]
 [-0.07561453 -0.01622762 -0.09162518 ...  0.00555181  0.07907773
   0.07408843]
 [-0.01447447 -0.07726149 -0.14450675 ... -0.06313405  0.01861974
  -0.02789287]
 ...
 [-0.12191684 -0.01099629 -0.12214984 ... -0.04715336 -0.06570611
  -0.01007605]
 [ 0.00521799  0.05436935 -0.08247132 ... -0.00235258  0.01250604
   0.06610787]
 [ 0.02201887  0.09483297 -0.16155362 ...  0.06904075 -0.01095848
  -0.05828862]]
FAISS index built.


Faiss Tests

In [ ]:
# Example: Search the index with a query embedding (optional)
index = faiss.read_index("./indices/my_faiss_index.idx")

query = "Whats the difference between DIVERSA premium and DIVERSA care in Outpatient treatments for orthodox medicine?"
query_embedding = compute_embeddings([query], tokenizer, model, device=device)
distances, indices = index.search(query_embedding.astype('float32'), k=5)
print("Top-5 results:", indices)

In [4]:
import os


Markitdown tests


In [ ]:
from markitdown import MarkItDown

md = MarkItDown()
result = md.convert("Docs/sample-1.pptx")
print(result.text_content)